In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load the dataset using the exact file name from your zip archive
df = pd.read_csv("GaltonFamilies.csv")
print(f"Dataset successfully loaded! Found {len(df)} records.")

Dataset successfully loaded! Found 934 records.


In [2]:
# ==========================================
# CELL 2: EM Algorithm & Live Prediction Engine
# ==========================================

import numpy as np

# 1. COMBINE DISTINCT POPULATIONS (Fathers and Children) AS INSTRUCTED
fathers = df['father'].values
children = df['childHeight'].values
combined_data = np.concatenate([fathers, children])

# -------------------------------
# INITIAL PARAMETERS
# -------------------------------
mu1, mu2 = 64.0, 69.0
var1, var2 = 4.0, 4.0
pi1, pi2 = 0.5, 0.5


# -------------------------------
# Gaussian PDF
# -------------------------------
def normal_pdf(x, mean, var):
    return (1 / np.sqrt(2 * np.pi * var)) * np.exp(-((x - mean) ** 2) / (2 * var))


# -------------------------------
# Log-Likelihood
# -------------------------------
def log_likelihood(data, mu1, mu2, var1, var2, pi1, pi2):
    likelihood = (
        pi1 * normal_pdf(data, mu1, var1)
        +
        pi2 * normal_pdf(data, mu2, var2)
    )
    return np.sum(np.log(likelihood))


# -------------------------------
# Tracking Table Header
# -------------------------------
print(f"{'Iteration':<10}"
      f"{'Mean 1':<10}"
      f"{'Mean 2':<10}"
      f"{'Variance 1':<12}"
      f"{'Variance 2':<12}"
      f"{'Pi1':<10}"
      f"{'Pi2':<10}"
      f"{'Log-Likelihood'}")

print("-"*95)


# ==========================================
# EM LOOP
# ==========================================
for iteration in range(3):

    # -------- E STEP --------
    r1 = pi1 * normal_pdf(combined_data, mu1, var1)
    r2 = pi2 * normal_pdf(combined_data, mu2, var2)

    total_r = r1 + r2

    w1 = r1 / total_r
    w2 = r2 / total_r

    ll = log_likelihood(
        combined_data,
        mu1,
        mu2,
        var1,
        var2,
        pi1,
        pi2
    )

    print(f"{iteration:<10}"
          f"{mu1:<10.2f}"
          f"{mu2:<10.2f}"
          f"{var1:<12.2f}"
          f"{var2:<12.2f}"
          f"{pi1:<10.2f}"
          f"{pi2:<10.2f}"
          f"{ll:.2f}")

    # -------- M STEP --------
    N1 = np.sum(w1)
    N2 = np.sum(w2)

    mu1 = np.sum(w1 * combined_data) / N1
    mu2 = np.sum(w2 * combined_data) / N2

    var1 = np.sum(w1 * (combined_data - mu1) ** 2) / N1
    var2 = np.sum(w2 * (combined_data - mu2) ** 2) / N2

    pi1 = N1 / len(combined_data)
    pi2 = N2 / len(combined_data)


# ==========================================
# SMART LIVE PREDICTION ENGINE
# ==========================================
def predict_live_height(height_input, unit="detect"):
    """
    Accepts any height input.
    If unit='detect', it automatically finds out the format:
       - Under 3.0  -> Treated as METERS (e.g., 1.81)
       - Over 90.0  -> Treated as CENTIMETERS (e.g., 181.0)
       - In between -> Treated as INCHES (e.g., 71.2)

    You can also force the unit by typing unit='inches', unit='cm', or unit='meters'
    """
    unit = unit.lower().strip()
    converted_inches = 0.0
    detected_unit = ""

    # ---- SMART UNIT DETECTION SYSTEM ----
    if unit == "detect":
        if height_input < 3.0:
            detected_unit = "Meters"
            cm = height_input * 100
            converted_inches = cm / 2.54
        elif height_input > 90.0:
            detected_unit = "Centimeters"
            converted_inches = height_input / 2.54
        else:
            detected_unit = "Inches"
            converted_inches = height_input

    # ---- FORCED UNIT OVERRIDES ----
    elif unit == "meters" or unit == "m":
        detected_unit = "Meters (Forced)"
        converted_inches = (height_input * 100) / 2.54
    elif unit == "cm":
        detected_unit = "Centimeters (Forced)"
        converted_inches = height_input / 2.54
    elif unit == "inches" or unit == "in":
        detected_unit = "Inches (Forced)"
        converted_inches = height_input

    # Posterior probability math calculations
    p1 = pi1 * normal_pdf(converted_inches, mu1, var1)
    p2 = pi2 * normal_pdf(converted_inches, mu2, var2)

    total = p1 + p2
    prob1 = p1 / total
    prob2 = p2 / total

    print("\n" + "="*50)
    print("      LIVE ENGINE OUTPUT (GROUP 12)")
    print("="*50)
    print(f"Original Input  : {height_input}")
    print(f"Detected Unit   : {detected_unit}")
    print(f"Converted Height: {converted_inches:.2f} inches")
    print(f"Probability Group 1 (Shorter Cohort - Children): {prob1*100:.2f}%")
    print(f"Probability Group 2 (Taller Cohort - Fathers)  : {prob2*100:.2f}%")

    if prob1 > prob2:
        print("Prediction: Group 1 (Shorter Cohort - Children)")
    else:
        print("Prediction: Group 2 (Taller Cohort - Fathers / Pros)")
    print("="*50)


# ==========================================
# PRESENTATION DAY TEST RUN EXAMPLES
# ==========================================
# Test 1: Typing in Meters
predict_live_height(1.51)

# Test 2: Typing in Inches
predict_live_height(71.5)

# Test 3: Typing in Centimeters
predict_live_height(185.0)

Iteration Mean 1    Mean 2    Variance 1  Variance 2  Pi1       Pi2       Log-Likelihood
-----------------------------------------------------------------------------------------------
0         64.00     69.00     4.00        4.00        0.50      0.50      -5091.68
1         64.43     69.71     4.78        4.80        0.33      0.67      -4867.73
2         64.47     69.66     5.10        5.03        0.33      0.67      -4864.56

      LIVE ENGINE OUTPUT (GROUP 12)
Original Input  : 1.51
Detected Unit   : Meters
Converted Height: 59.45 inches
Probability Group 1 (Shorter Cohort - Children): 99.89%
Probability Group 2 (Taller Cohort - Fathers)  : 0.11%
Prediction: Group 1 (Shorter Cohort - Children)

      LIVE ENGINE OUTPUT (GROUP 12)
Original Input  : 71.5
Detected Unit   : Inches
Converted Height: 71.50 inches
Probability Group 1 (Shorter Cohort - Children): 0.66%
Probability Group 2 (Taller Cohort - Fathers)  : 99.34%
Prediction: Group 2 (Taller Cohort - Fathers / Pros)

      LIVE